# Processamento de Linguagem Natural Desmistificado | TF-IDF

Material baseado no curso *NLP Demystified*:<br>
https://nlpdemystified.org<br>
https://github.com/futuremojo/nlp-demystified

---

Na **Aula 02** vimos como transformar texto em números com o **Bag-of-Words (BoW)** e como medir a similaridade entre documentos com a **similaridade do cosseno**. Nesta aula damos o próximo passo: a ponderação **TF-IDF**, que corrige uma limitação importante do BoW por frequência, e a usamos para construir um pequeno **motor de busca**.

O melhor: em vez de um corpus de brinquedo, vamos trabalhar com os **dados que vocês mineraram** — uma coleção de *abstracts* (resumos) de artigos científicos exportada da **Web of Science (Clarivate)** ou da **OpenAlex**.

### Atualização do spaCy e instalação de pacotes

Na época em que este notebook foi criado, o spaCy já possuía versões mais recentes, mas o Google Colab ainda vinha com a versão 2.x por padrão. Por isso, o primeiro passo é **atualizar o spaCy** e **baixar um modelo estatístico de língua** (aqui, o modelo de inglês `en_core_web_sm`).

> **O que é o spaCy?** É uma das bibliotecas mais populares de PLN em Python. Ela cuida de tarefas de baixo nível como *tokenização* (dividir o texto em palavras/tokens), *lematização* (reduzir a palavra à sua forma canônica), *marcação morfossintática* (POS tagging) e *reconhecimento de entidades nomeadas* (NER). Usaremos o spaCy como "tokenizador" personalizado mais adiante.

<br>

**IMPORTANTE**<br>
Se você estiver rodando isto na nuvem (Colab) em vez de um servidor Jupyter local, o notebook irá **desconectar** (*timeout*) após um período de inatividade. Se isso acontecer e você não reconectar a tempo, será necessário atualizar o spaCy novamente e reinstalar os pacotes estatísticos.

<br>

Veja este link sobre como executar notebooks do Colab localmente na sua máquina para evitar esse problema:<br>
https://research.google.com/colaboratory/local-runtimes.html

In [ ]:
!pip install spacy==3.*
!python -m spacy download en_core_web_sm
!python -m spacy info

In [ ]:
!pip install scikit-learn --upgrade

# TF-IDF

Módulo do curso para esta demonstração: https://www.nlpdemystified.org/course/tf-idf

---

O BoW por frequência tem um problema: palavras muito comuns (como "the", "of", "de", "a") aparecem em quase todos os documentos e acabam recebendo contagens altas, **sem** ajudar a distinguir um documento do outro. O **TF-IDF** corrige isso ponderando cada termo por duas quantidades:

* **TF — *Term Frequency* (frequência do termo):** quão frequente é o termo *dentro* de um documento. Quanto mais um termo aparece em um documento, mais ele o caracteriza.
* **IDF — *Inverse Document Frequency* (frequência inversa nos documentos):** quão *raro* é o termo *no corpus inteiro*. Termos que aparecem em pouquíssimos documentos são muito informativos; termos presentes em quase todos são pouco informativos.

O peso final é o **produto** `TF × IDF`. Assim, um termo recebe peso alto quando é frequente em *um* documento **e** raro no corpus como um todo — exatamente as palavras que melhor identificam o assunto daquele documento. O IDF costuma ser calculado com um logaritmo:

$$\text{idf}(t) = \log\frac{N}{\text{df}(t)}$$

onde $N$ é o número de documentos e $\text{df}(t)$ é o número de documentos que contêm o termo $t$.

In [ ]:
import spacy
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Carregando os dados minerados (Web of Science / OpenAlex)

Em vez de baixar um conjunto de dados pronto da internet, vamos usar o corpus que **vocês mesmos mineraram**: os *abstracts* dos artigos exportados da **Web of Science (Clarivate)** ou da **OpenAlex**.

No exemplo abaixo usamos o arquivo `savedrecs1.xls` (uma exportação da Web of Science com 1.000 artigos sobre *high-entropy alloys*). **Troque esse arquivo pelo que você exportou** no seu projeto.

> **Dica sobre o nome da coluna:** em exportações da Web of Science, a coluna com o resumo costuma se chamar `Abstract`. Se você usou a OpenAlex (ou outro nome de coluna), ajuste o código para apontar para a coluna correta.

In [ ]:
# Instale o leitor de arquivos .xls, caso ainda não o tenha.
!pip install xlrd

import pandas as pd

# Carregue a planilha exportada da Web of Science (Clarivate) ou da OpenAlex.
# Troque 'savedrecs1.xls' pelo arquivo que VOCÊ minerou.
dados = pd.read_excel('savedrecs1.xls')

# Usaremos a coluna 'Abstract' (o resumo de cada artigo) como nosso corpus.
# Convertemos para uma LISTA de strings: cada elemento é um documento.
corpus = dados['Abstract'].tolist()

print(f'Número de artigos (documentos) no corpus: {len(corpus)}')

### Inspecionando o corpus

Antes de vetorizar, vale a pena dar uma olhada na estrutura e no conteúdo do nosso corpus.

In [ ]:
# Nosso corpus é uma lista de strings, em que cada string é um abstract.
print(type(corpus))

In [ ]:
# Número de documentos no corpus.
len(corpus)

In [ ]:
# Vamos olhar os dois primeiros abstracts.
corpus[:2]

## Criando features TF-IDF

In [ ]:
# Como antes, se quisermos usar o tokenizador do spaCy, precisamos criar uma
# função de callback. Lembre-se de atualizar o spaCy se necessário (consulte
# o início do arquivo para comentários e instruções).
nlp = spacy.load('en_core_web_sm')

# Não precisamos de reconhecimento de entidades nomeadas (NER) nem de análise de
# dependências (parser) para esta tarefa, então desabilitamos esses componentes.
# Isso acelera o pipeline. Precisamos, porém, da marcação morfossintática (POS
# tagging), pois ela é necessária para a lematização.
unwanted_pipes = ["ner", "parser"]

# Para este exercício, vamos remover pontuação e espaços (o que inclui quebras de
# linha), filtrar apenas os tokens compostos por caracteres alfabéticos e retornar
# o lema (lemma_), que depende da marcação morfossintática (POS tagging).
def spacy_tokenizer(doc):
  with nlp.disable_pipes(*unwanted_pipes):
    return [t.lemma_ for t in nlp(doc) if \
            not t.is_punct and \
            not t.is_space and \
            t.is_alpha]

Assim como existem classes para criar vetores BoW de frequência bruta e binários, o scikit-learn inclui uma classe análoga chamada **TfidfVectorizer** para criar vetores **TF-IDF** a partir de um corpus.<br>
https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

<br>

O padrão de uso é o mesmo do `CountVectorizer`: chamamos *fit_transform* sobre o corpus, o que gera o dicionário de vocabulário (etapa *fit*) e produz os vetores TF-IDF (etapa *transform*). A grande diferença está **no que cada célula da matriz guarda**: em vez de uma contagem inteira, teremos um **peso TF-IDF** (número real) refletindo a importância do termo naquele documento.

In [ ]:
%%time
# Usar as configurações padrão do TfidfVectorizer.
vectorizer = TfidfVectorizer(tokenizer=spacy_tokenizer)
features = vectorizer.fit_transform(corpus)

In [ ]:
# O número de tokens únicos.
print(len(vectorizer.get_feature_names_out()))

In [ ]:
# As dimensões da nossa matriz de features: X linhas (documentos) por Y colunas (tokens).
print(features.shape)

In [ ]:
# Como fica a codificação do primeiro documento no formato esparso.
print(features[0])

Como mencionamos nos slides, existem **variações** do TF-IDF, e o scikit-learn, entre outras coisas, aplica **suavização** (*smoothing* — soma 1 ao numerador e ao denominador do componente IDF, para evitar divisão por zero e amortecer pesos extremos) e, por padrão, **normaliza** os vetores (cada vetor de documento passa a ter norma 1). Ambos os comportamentos podem ser desativados, se desejado, pelos parâmetros *smooth_idf* e *norm*, respectivamente. Veja mais informações aqui:<br>
https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

> **Por que normalizar?** A normalização faz com que documentos de tamanhos diferentes fiquem em pé de igualdade e, na prática, combina muito bem com a similaridade do cosseno (que, lembre-se, já é insensível à magnitude dos vetores).

## Consultando os dados (*querying*)

As técnicas de medição de similaridade que aprendemos antes podem ser usadas aqui da mesma forma. Na prática, conseguimos **consultar** (fazer uma *busca* no) nosso corpus com esta sequência de passos:

1. **Transformar** a consulta (*query*) usando o **mesmo vocabulário** aprendido na etapa *fit* sobre o corpus. (Por isso chamamos apenas `transform`, e não `fit_transform`: a consulta precisa "morar" no mesmo espaço vetorial dos documentos.)
2. Calcular as **similaridades do cosseno** entre a consulta e cada documento do corpus.
3. **Ordenar** os documentos em ordem decrescente de pontuação.

O documento com maior similaridade é o "melhor resultado" da busca — é exatamente essa a ideia por trás de um motor de busca baseado em TF-IDF.

In [ ]:
# Transformar a consulta em um vetor TF-IDF.
query = ["largest bulk modulus of a high entropy alloy"]
query_tfidf = vectorizer.transform(query)

In [ ]:
# Calcular as similaridades do cosseno entre a consulta e cada documento.
# Chamamos flatten() aqui porque o cosine_similarity retorna uma lista de
# listas e queremos apenas uma lista simples.
cosine_similarities = cosine_similarity(features, query_tfidf).flatten()

Agora que temos a lista de similaridades do cosseno, podemos usar esta função utilitária para retornar os **índices dos *k* documentos** com as maiores similaridades — ou seja, os *k* resultados mais relevantes para a consulta.

In [ ]:
import numpy as np

# O método argsort() do NumPy retorna uma lista de *índices* que
# ordenariam o array:
# https://numpy.org/doc/stable/reference/generated/numpy.argsort.html
#
# A ordenação é crescente, mas queremos as k maiores similaridades do cosseno,
# que ficam no final da ordenação. Por isso negamos k e pegamos as últimas k
# entradas da lista de índices, em ordem reversa. Existem formas mais rápidas
# de fazer isso (por exemplo, com argpartition), mas esta é mais concisa.
def top_k(arr, k):
  kth_largest = (k + 1) * -1
  return np.argsort(arr)[:kth_largest:-1]

In [ ]:
# Então, para a consulta acima, estes são os cinco documentos mais relevantes.
top_related_indices = top_k(cosine_similarities, 5)
print(top_related_indices)

In [ ]:
# Vamos dar uma olhada nas respectivas similaridades do cosseno.
print(cosine_similarities[top_related_indices])

In [ ]:
# Melhor resultado (top match).
print(corpus[top_related_indices[0]])

In [ ]:
# Segundo melhor resultado.
print(corpus[top_related_indices[1]])

In [ ]:
# Vamos tentar uma consulta diferente.
query = ["stability of noble metal high entropy alloys"]
query_tfidf = vectorizer.transform(query)

cosine_similarities = cosine_similarity(features, query_tfidf).flatten()
top_related_indices = top_k(cosine_similarities, 5)

print(top_related_indices)
print(cosine_similarities[top_related_indices])

In [ ]:
print(corpus[top_related_indices[0]])

Pronto! Aqui temos o **esqueleto de um motor de busca simples** — mas ainda estamos muito longe de competir com buscadores comerciais de prateleira, e mais ainda do Google.

<br>

- Para cada consulta, estamos varrendo o corpus **inteiro**. Na prática, você vai querer construir um **índice invertido** (*inverted index*), que mapeia cada termo diretamente aos documentos que o contêm, tornando a busca muito mais rápida. Aplicações como o **Elasticsearch** fazem isso "por baixo dos panos".
- Você também vai querer **avaliar a qualidade** da sua busca usando métricas como **precisão** (*precision*) e **revocação/recall** (*recall*).
- O **ranqueamento** dos documentos costuma ser bem mais sofisticado, usando funções de *ranking* como a **Okapi BM25**. Nos grandes buscadores, o ranqueamento envolve centenas de variáveis: o que o usuário buscou antes, no que ele costuma clicar, onde ele está fisicamente, e assim por diante. Essas variáveis fazem parte do "molho secreto" e são zelosamente guardadas pelas empresas.
- Além da mera **presença de palavras**, a **intenção** e o **significado** vêm ganhando um papel cada vez maior (é aqui que entram, mais adiante no curso, os *embeddings* e os modelos de linguagem).

<br>

**Recuperação de Informação** (*Information Retrieval*) é um tema enorme e rico. Para além da busca, ele também é peça-chave em tarefas como **resposta a perguntas** (*question answering*).

## Exercícios — TF-IDF

**EXERCÍCIO**<br>
Se tiver curiosidade, aprofunde-se nos conceitos que acabamos de mencionar.<br>

- Índice invertido: https://en.wikipedia.org/wiki/Inverted_index<br>
- Precisão e revocação (precision/recall): https://en.wikipedia.org/wiki/Precision_and_recall<br>
- Okapi BM25: https://en.wikipedia.org/wiki/Okapi_BM25<br>

In [ ]:
#
# EXERCÍCIO: repita esta análise usando OUTRO conjunto de dados que você
# mesmo minerou na Web of Science (Clarivate) ou na OpenAlex — por exemplo,
# artigos de um tema diferente, ligado ao seu projeto. Basta trocar o arquivo
# .xls carregado no início e reexecutar o notebook.
#
# Faça algumas consultas (queries) relevantes ao seu tema e verifique se os
# documentos recuperados fazem sentido.
#
# Se estiver se sentindo ambicioso:
#  - incorpore n-gramas (parâmetro ngram_range do TfidfVectorizer);
#  - remova stop words no seu tokenizador;
#  - pesquise como medir a qualidade da busca com precisão (precision) e
#    revocação (recall).
#